# CIRA-CIC-DoHBrw-2020 validation of the proposed detector

Goal 2 of the project: validate the proposed **time-distributed dilated conv → BiLSTM → residual-fusion** model on the public CIRA-CIC-DoHBrw-2020 dataset and compare with published results and with RF / XGBoost / MLP trained on the identical split.

Sections: **0 setup (Drive)** → **1 download dataset to Drive** → **2 inspect** → **3 per-flow protocol (T=1)** → **4 persistent-session windows (T=8) + ablation** → **5 binary task** → **6 collect results**.

Runtime: GPU (T4) recommended for sections 3–5.

In [ ]:
# ===== 0. Setup: Drive + repo =====
import os, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount failed/skipped:', e); os.makedirs('/content/drive/MyDrive', exist_ok=True)
DS = '/content/drive/MyDrive/doq-dataset'
CIRA = f'{DS}/cira-dohbrw-2020'          # official Total-CSVs (l1-doh, l1-nondoh, l2-benign, l2-malicious)
CIRA_TOOL = f'{DS}/cira-dohbrw-2020-bytool'  # per-tool CSVs (mal-dns2tcp / mal-dnscat2 / mal-iodine) + l2-benign
RES = f'{DS}/results/cira'                # results copied here
os.makedirs(CIRA, exist_ok=True); os.makedirs(RES, exist_ok=True)

REPO_URL = 'https://github.com/Cyberguru1/doq-tunnel-dataset.git'
if not os.path.isdir('/content/doq-tunnel-dataset'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/doq-tunnel-dataset'], check=True)
else:
    subprocess.run(['git', '-C', '/content/doq-tunnel-dataset', 'pull'], check=True)
os.chdir('/content/doq-tunnel-dataset')
subprocess.run(['pip', 'install', '-q', 'xgboost', 'tabulate', 'kagglehub'], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('repo at', os.getcwd()); print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# ===== 1. Download CIRA-CIC-DoHBrw-2020 to Drive (original, unmodified CIC CSVs) =====
# Official host https://cicresearch.ca/CICDataset/DoHBrw-2020/Dataset/ now redirects to a UNB sign-up page, so we use
# Kaggle mirrors of the ORIGINAL files (the dataset licence allows mirroring):
#   dhoogla/cicdohbrw2020 v1            -> Total-CSVs: l1-doh.csv 269,643 / l1-nondoh.csv 897,493 / l2-benign.csv 19,807 / l2-malicious.csv 249,836 rows
#   peterfriedrich1/dns-test-traffic-dohbrw2020 -> per-tool: mal-dns2tcp.csv / mal-dnscat2.csv / mal-iodine.csv (+ benign-chrome/firefox)
# Row counts are verified against the official release in PROGRESS_LOG.md (2026-11-11).
!python src/model/fetch_cira.py {CIRA} dhoogla/cicdohbrw2020/versions/1
!python src/model/fetch_cira.py {CIRA_TOOL} peterfriedrich1/dns-test-traffic-dohbrw2020
!cp -n {CIRA}/l2-benign.csv {CIRA_TOOL}/
!wc -l {CIRA}/*.csv {CIRA_TOOL}/*/*.csv

In [ ]:
# ===== 2. Inspect the files (columns, rows, labels) =====
import pandas as pd
for f in sorted(glob.glob(f'{CIRA}/**/*', recursive=True) + glob.glob(f'{CIRA_TOOL}/**/*', recursive=True)):
    if not os.path.isfile(f): continue
    try:
        df = pd.read_parquet(f) if f.endswith('.parquet') else pd.read_csv(f, low_memory=False)
    except Exception as e:
        print('skip', f, e); continue
    lab = df['Label'].value_counts().to_dict() if 'Label' in df.columns else '-'
    print(f'{os.path.relpath(f, CIRA):40s} rows={len(df):8d} cols={df.shape[1]:3d} labels={lab}')
print(list(df.columns))

In [ ]:
# ===== 3. Tool identification, per-flow protocol (T=1, 80/10/10 stratified) + RF/XGB/MLP on the same split =====
!python src/model/train.py --dataset cira --root {CIRA_TOOL} --task tool --seq-len 1 --epochs 60 --patience 12 --baselines --out results/cira_tool_T1 2>&1 | grep -v '^epoch'

In [ ]:
# ===== 4. Persistent-session windows: the proposed model (PLE feature tokens + sliding-window training) =====
# Final protocol: windows of T consecutive flows of one client->resolver session (non-overlapping for val/test),
# 5-fold stratified CV, RF/XGB/MLP trained on the flattened T x 29 windows of the identical folds.
for T, S in ((16, 4), (32, 8)):
    !python src/model/train.py --dataset cira --root {CIRA_TOOL} --task tool --seq-len {T} --train-stride {S} --cv 5 --epochs 80 --patience 15 --feat-embed ple --sched cosine --baselines --out results/cira_tool_T{T}_ts{S}_ple_cv5 2>&1 | grep -v '^epoch'
# Ablations (single split): no PLE, no sliding windows, base variant (no residual fusion), group split (unseen sessions)
!python src/model/train.py --dataset cira --root {CIRA_TOOL} --task tool --seq-len 16 --epochs 80 --patience 15 --out results/cira_tool_T16 2>&1 | grep -v '^epoch'
!python src/model/train.py --dataset cira --root {CIRA_TOOL} --task tool --seq-len 16 --epochs 80 --patience 15 --feat-embed ple --sched cosine --out results/cira_tool_T16_ple 2>&1 | grep -v '^epoch'
!python src/model/train.py --dataset cira --root {CIRA_TOOL} --task tool --seq-len 16 --train-stride 4 --epochs 80 --patience 15 --feat-embed ple --sched cosine --variant base --out results/cira_tool_T16_ts4_ple_base 2>&1 | grep -v '^epoch'
!python src/model/train.py --dataset cira --root {CIRA_TOOL} --task tool --seq-len 16 --train-stride 4 --split group --epochs 80 --patience 15 --feat-embed ple --sched cosine --baselines --out results/cira_tool_T16_ts4_ple_group 2>&1 | grep -v '^epoch'

In [ ]:
# ===== 5. Binary benign-DoH vs malicious-DoH (layer 2): per-flow and session windows =====
!python src/model/train.py --dataset cira --root {CIRA} --task binary --seq-len 1 --epochs 30 --baselines --out results/cira_bin_T1 2>&1 | grep -v '^epoch'
!python src/model/train.py --dataset cira --root {CIRA} --task binary --seq-len 16 --train-stride 4 --epochs 60 --patience 15 --feat-embed ple --sched cosine --baselines --out results/cira_bin_T16_ts4_ple 2>&1 | grep -v '^epoch'

In [ ]:
# ===== 6. Collect reports + comparison table -> Drive =====
import shutil
for d in sorted(glob.glob('results/cira_*')):
    shutil.copytree(d, os.path.join(RES, os.path.basename(d)), dirs_exist_ok=True)
    print('\n' + open(os.path.join(d, 'report.md')).read())
!python src/model/compare.py results/cira_* | tee {RES}/comparison.md
print('saved to', RES)